In [ ]:
# Cell 0 - settings
MODEL_NAME = "run1"   # the model chosen in Phase 2
CKPT = "auto"         # "auto" = the one attached input that has checkpoints/best.pt; or a path
OUT = "/kaggle/working/phase3"

# run1's validation numbers from docs/experiments.md, shown next to the test score
VAL = {"mae": 11.233, "rmse": 19.280, "median_ae": 5.713, "smape": 53.513, "bias": -4.345}
print(f"Phase 3 for {MODEL_NAME}, outputs in {OUT}")

In [ ]:
# Cell 1 - environment. No GPU needed: Accelerator "None" is fine for this notebook.
import os
import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.is_available(), "| CPUs:", os.cpu_count())

In [ ]:
# Cell 2 - helper, code, packages
import os
import subprocess
import sys
from shlex import quote as q  # quotes paths for the shell


def run(cmd):
    """Run a shell command, show its output live, and STOP the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit code {p.returncode}): {cmd}")


REPO = "/tmp/repo"
run(f"rm -rf {REPO} && git clone -q -b feature/model-evaluation --depth 1 "
    f"https://github.com/Akash7762/Amazon-Vision-Based-Price-Predictor.git {REPO}")
os.chdir(REPO)
sys.path.insert(0, REPO)
run("git log --oneline -1")
run("test -f model/export.py && test -f model/error_analysis.py && echo 'code is up to date' "
    "|| (echo 'OLD CODE on GitHub: push feature/model-evaluation from your PC first' && exit 1)")

run('pip install -q "timm>=1.0.27" onnx onnxruntime onnxscript')
run('python -c "import torch, timm, onnx, onnxruntime; '
    'print(\'torch\', torch.__version__, \'| timm\', timm.__version__, '
    '\'| onnx\', onnx.__version__, \'| onnxruntime\', onnxruntime.__version__)"')

In [ ]:
# Cell 3 - data, images and the chosen model's checkpoint
import glob
import pandas as pd

META = "/kaggle/input/datasets/akashverma7762/amazon-price-metadata/metadata.csv"
IMG = "/kaggle/input/datasets/akashverma7762/amazon-price-images-v2/images"

if not (os.path.exists(META) and os.path.isdir(IMG)):
    # Fallback search at dataset-folder level only; never lists the 73k-image folder.
    metas = glob.glob("/kaggle/input/*/metadata.csv") + glob.glob("/kaggle/input/datasets/*/*/metadata.csv")
    imgs = glob.glob("/kaggle/input/*/images") + glob.glob("/kaggle/input/datasets/*/*/images")
    print("found metadata:", metas)
    print("found images:  ", imgs)
    assert len(metas) == 1 and len(imgs) == 1, (
        "Attach exactly one metadata and one images dataset: amazon-price-metadata and "
        "amazon-price-images-v2.")
    META, IMG = metas[0], imgs[0]

ids = set(pd.read_csv(META)["sample_id"].astype(str))
have = {f[:-4] for f in os.listdir(IMG) if f.endswith(".jpg")}  # ~1 min, the folder is big
print(f"META = {META}\nIMG  = {IMG}\nimages={len(have)} missing={len(ids - have)}")
assert len(ids - have) < 100, "Images missing - wrong images dataset attached?"

if CKPT == "auto":
    # <input>/.../checkpoints/best.pt up to 4 folders deep (deeper would list the image folder).
    hits = sorted({h for depth in range(1, 5)
                   for h in glob.glob("/kaggle/input/" + "*/" * depth + "checkpoints/best.pt")})
    print("checkpoints found:", hits)
    assert len(hits) == 1, (
        f"Expected exactly one attached checkpoints/best.pt, found {len(hits)}. Attach only "
        f"{MODEL_NAME}'s output (amazon-price-run1), or set CKPT in Cell 0 to a path.")
    CKPT = hits[0]
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
print(f"CKPT = {CKPT}\n  epoch {ck['epoch']}, target {ck.get('args', {}).get('target', 'price')}, "
      f"val_loss {ck['val_loss']:.4f}")
del ck

In [ ]:
# Cell 4 - the test score: once, for the chosen model (~27 min on CPU)
import json

run(f"python -u model/evaluate.py --checkpoint {q(CKPT)} --name {MODEL_NAME} "
    f"--metadata {q(META)} --image-dir {q(IMG)} --split test --num-workers 2 "
    f"--out-dir {q(OUT + '/eval')}")

test = json.load(open(f"{OUT}/eval/{MODEL_NAME}_test.json"))
print(f"\n| | val (used for choosing) | test (never used before) |\n|---|---|---|")
for key, label in [("mae", "MAE ($)"), ("rmse", "RMSE ($)"), ("median_ae", "Median abs error ($)"),
                   ("smape", "SMAPE (%)"), ("bias", "Bias ($)")]:
    print(f"| {label} | {VAL[key]:.3f} | {test['model'][key]:.3f} |")
lo, hi = test["model_ci95"]["mae"]
print(f"\nTest MAE ${test['model']['mae']:.3f} (95% interval ${lo:.3f} to ${hi:.3f}); "
      f"median-price baseline on test ${test['baseline_median']['mae']:.3f}")

In [ ]:
# Cell 5 - error analysis on the test predictions (~1 min)
from IPython.display import Image as Show, display

run(f"python -u model/error_analysis.py "
    f"--predictions {q(OUT + '/eval/' + MODEL_NAME + '_test_predictions.csv')} "
    f"--metadata {q(META)} --image-dir {q(IMG)} --out-dir {q(OUT + '/analysis')} --name {MODEL_NAME}")
for f in ["calibration.png", "worst_under.jpg", "worst_over.jpg"]:
    display(Show(filename=f"{OUT}/analysis/{f}"))

In [ ]:
# Cell 6 - export to ONNX, and check it against PyTorch on 256 test images (~2-3 min)
run(f"python -u model/export.py --checkpoint {q(CKPT)} --out-dir {q(OUT + '/model')} "
    f"--metadata {q(META)} --image-dir {q(IMG)} --verify-split test --verify-n 256")

In [ ]:
# Cell 7 - the backend's route: re-price 8 test photos with only onnxruntime
# (no PyTorch), and check they match Cell 4's prices to the cent.
run(f"python -u model/predict_onnx.py --model {q(OUT + '/model/price_model.onnx')} "
    f"--check-predictions {q(OUT + '/eval/' + MODEL_NAME + '_test_predictions.csv')} "
    f"--image-dir {q(IMG)} --n 8")

In [ ]:
# Cell 8 - what to download
total = 0
for f in sorted(glob.glob(f"{OUT}/**", recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        total += size
        print(f"{size / 1e6:8.1f} MB  {f}")
print(f"{total / 1e6:8.1f} MB  total")
card = json.load(open(f"{OUT}/model/price_model.json"))
print(f"\nprice_model.onnx sha256 {card['sha256']}")
print(f"verification: {card['verification']['n_images']} test images, max difference "
      f"${card['verification']['max_abs_diff_usd']:.6f} -> "
      f"{'PASSED' if card['verification']['passed'] else 'FAILED'}")